# Koopman-MPC vs bases sérieuses — protocole rigoureux (v2)

**Question de recherche reposée.** Quand un modèle de Koopman *identifié sur données*
(avec observables adaptées) apporte-t-il un avantage mesurable à un MPC, par rapport à
(B1) un **PID géométrique + DOB réglé**, (B2) un **MPC à modèle physique nominal + DOB
(offset-free)**, (C1) le même Koopman-MPC avec le **rayon spectral de A contraint** (Schur),
(C2) un **retour d'état LQR** (horizon infini) sur ce modèle Koopman stabilisé, et (C3) le
même LQR construit directement sur le modèle Koopman **brut** (non stabilisé) — sous
*discordance de modèle structurelle* (traînée quadratique, retard/lag des moteurs, décalage
du centre de masse, rafales, changement de masse) ? **6 contrôleurs comparés au total.**

**Pourquoi C3 (LQR sur modèle brut) a été ajouté.** Un premier run complet (5 contrôleurs,
100 essais) a montré que la stabilisation spectrale (C1) dégrade catastrophiquement le
contrôle : 70 % d'échec avec DOB, 85 % sans DOB. Le LQR construit sur ce même modèle stabilisé
(C2) échouait aussi (58 %), mais de façon **confondue** : impossible de savoir si c'était la
stratégie LQR elle-même ou le modèle cassé qui était en cause. C3 isole la question en gardant
la stratégie LQR mais en revenant au modèle Koopman brut. Un test préalable confirme que
l'équation de Riccati discrète converge malgré rho(A) > 1 en boucle ouverte (la paire (A,B)
reste stabilisable : rho(A-BK) = 0,987 avec les poids par défaut), donc C3 est numériquement
valide et répond proprement à « LQR vs MPC », indépendamment de « modèle stabilisé vs brut ».

**Ce que cette version change (par rapport à la v1 « Koopman adaptatif »)**

| Point | Réponse dans ce code |
|---|---|
| 1. Question / modèle | Dictionnaire *non linéaire et physiquement motivé* (‖v‖v pour la traînée, retards d'entrée pour le lag moteur). Le gain du modèle est mesuré en boucle ouverte à 1/5/10/20 pas (plantes *hors* identification) et le dictionnaire est choisi sur la validation, pas sur le test. |
| 2. Bases sérieuses | B1 : PID+DOB **réglé** par recherche aléatoire (pas de gains « à la main »). B2 : MPC physique+DOB, même solveur, mêmes poids, même horizon que le Koopman → seule la **source du modèle** change. |
| 3. Hors simulateur d'identification | Plantes de test tirées avec des paramètres **jamais vus** (m, J, traînée, lag, retard, CoM), régimes de perturbation absents des données d'identification, second régime « idéal » où Koopman ne peut pas gagner par construction. **Pas d'essais réels : à dire explicitement dans le papier.** |
| 4. Perturbations riches | Rafales (Ornstein-Uhlenbeck), rampes de vent, composante sinusoïdale, échelon de masse, CoM décalé, traînée, trajectoires `circle` / `figure8` / `agile`. |
| 5. Protocole propre | Découpage `tune` / `val` / `test` / `abl` avec graines disjointes ; hyper-paramètres **gelés et sauvegardés avant le test** ; ≥ 100 essais de test ; **les divergences sont une métrique** (taux d'échec + RMSE pénalisé, aucune exclusion) ; Wilcoxon apparié + IC bootstrap + correction de Holm. |

**Limites à écrire dans le papier :** simulation uniquement ; l'identification utilise des états propres (lissage hors ligne supposé) ; le résultat dépend du régime de discordance choisi (d'où le régime « idéal » de contrôle) ; hyper-paramètres réglés sur peu d'essais.

## Installation (Colab)

Exécutez d'abord la cellule ci-dessous. Ensuite, exécutez les cellules dans l'ordre.

**Durée du run complet** (`FAST_MODE = False`) : 6 contrôleurs principaux à régler et tester désormais
(contre 5 au run précédent, qui avait pris 64,5 minutes). Comptez **70 à 130 minutes** selon le nombre de
cœurs alloués par Colab.

**Sauvegarde** : Google Drive est monté automatiquement (`/content/drive/MyDrive/koopman_mpc_v2`). Une
archive ZIP de tout le dossier est créée à la fin et peut être téléchargée directement.

In [ ]:
# Dépendances non pré-installées sur Colab
!pip install -q osqp
print("osqp installé.")

In [ ]:
# =============================================================================
# 0. SETUP ET CONFIGURATION
# =============================================================================
import os, sys, time, json, math, warnings, shutil, itertools
from pathlib import Path
import multiprocessing as mp
import numpy as np, pandas as pd
import matplotlib
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.linalg import block_diag, schur
from scipy.stats import wilcoxon
from scipy.linalg import solve_discrete_are
import osqp
warnings.filterwarnings("ignore")

SEED = 20261001
FAST_MODE = False            # True = test de fumée (quelques minutes)

DT, SUBSTEPS = 0.05, 5
DT_SIM = DT / SUBSTEPS
G = 9.81
N_H = 20

if FAST_MODE:
    T_DUR, N_TEST, N_TEST_IDEAL, N_ABL = 10.0, 8, 4, 4
    N_TUNE_TR, N_TUNE_PID, N_VAL, N_EP_ID = 4, 4, 2, 10
    N_BOOT = 300
else:
    T_DUR, N_TEST, N_TEST_IDEAL, N_ABL = 30.0, 100, 32, 32
    N_TUNE_TR, N_TUNE_PID, N_VAL, N_EP_ID = 8, 40, 8, 40
    N_BOOT = 3000
N_STEPS_ID = 200

RMSE_CAP  = 1.0      # m : RMSE imputé pour un essai en échec (RMSE pénalisé)
FAIL_ERR  = 3.0      # m : erreur de position > seuil => échec (perte de suivi)
FAIL_W    = 20.0     # rad/s
LAMBDA_U  = 1e-3     # poids de ControlVariation dans le score de réglage

N_JOBS = max(1, (os.cpu_count() or 1) - 0) if sys.platform.startswith("linux") else 1
N_JOBS = int(os.environ.get("N_JOBS", N_JOBS))

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive') / "koopman_mpc_v2"
else:
    PROJECT_ROOT = Path("./koopman_mpc_v2")
FIG_DIR, TAB_DIR, DATA_DIR = PROJECT_ROOT/"figures", PROJECT_ROOT/"tables", PROJECT_ROOT/"data"
for d in (PROJECT_ROOT, FIG_DIR, TAB_DIR, DATA_DIR): d.mkdir(parents=True, exist_ok=True)
print(f"Résultats : {PROJECT_ROOT.resolve()} | FAST={FAST_MODE} | N_JOBS={N_JOBS}")

# Actionneurs et poids MPC (identiques pour tous les MPC)
T_MAX = 20.0
TAU_MAX = np.array([0.5, 0.5, 0.25])
U_MIN = np.r_[0.0, -TAU_MAX]; U_MAX = np.r_[T_MAX, TAU_MAX]
U_HOVER = np.array([1.0*G, 0., 0., 0.])
Q  = np.diag([120., 120., 160.]); Qv = np.diag([8., 8., 10.])
Qt = np.diag([30., 30., 30.]);   Qw = np.diag([1.0, 1.0, 0.2])
R_U = np.diag([0.008, 0.08, 0.08, 0.04]); RD_U = 0.025*np.eye(4)
QN_FACTOR = 5.0

DOB_D_MAX = 5.0
RHO_MAX = 1.0     # rayon spectral cible pour la variante Koopman stabilisée (koopstab)

In [ ]:
# =============================================================================
# 1. SO(3), RÉFÉRENCES
# =============================================================================
def hat(w):
    return np.array([[0., -w[2], w[1]], [w[2], 0., -w[0]], [-w[1], w[0], 0.]])

def project_so3(Rm):
    U, _, Vt = np.linalg.svd(Rm); Rn = U @ Vt
    if np.linalg.det(Rn) < 0:
        U[:, -1] *= -1; Rn = U @ Vt
    return Rn

def expm_so3(w):
    th = np.linalg.norm(w)
    if th < 1e-10: return np.eye(3) + hat(w)
    K = hat(w)
    return np.eye(3) + np.sin(th)/th*K + (1-np.cos(th))/th**2*K@K

def state_unpack(x): return x[0:3], x[3:6], x[6:15].reshape(3, 3), x[15:18]
def state_pack(p, v, Rm, w): return np.r_[p, v, Rm.reshape(-1), w]

def _fig8(t, w):
    s, c, wz = np.sin(w*t), np.cos(w*t), 0.125
    p = np.array([2*s, 2*s*c, 1.5+0.3*np.sin(wz*t)])
    v = np.array([2*w*c, 2*w*(c*c-s*s), 0.3*wz*np.cos(wz*t)])
    a = np.array([-2*w*w*s, -8*w*w*s*c, -0.3*wz*wz*np.sin(wz*t)])
    return p, v, a

def reference(t, family='circle'):
    if family == 'circle':
        p = np.array([2*np.cos(0.25*t), 2*np.sin(0.25*t), 1.5+0.3*np.sin(0.125*t)])
        v = np.array([-0.5*np.sin(0.25*t), 0.5*np.cos(0.25*t), 0.0375*np.cos(0.125*t)])
        a = np.array([-0.125*np.cos(0.25*t), -0.125*np.sin(0.25*t), -0.0046875*np.sin(0.125*t)])
        return p, v, a
    if family == 'figure8': return _fig8(t, 0.25)
    if family == 'agile':   return _fig8(t, 0.50)      # 2x plus rapide, a_max ~ 1 m/s^2
    raise ValueError(family)

In [ ]:
# =============================================================================
# 2. PLANTE « VÉRITÉ » (traînée, lag moteur, retard, CoM, rafales, masse)
# =============================================================================
# theta : m, J(3), Drot(3), cd (traînée quadratique), cl (linéaire), kw (force du vent),
#         lag (bool), tauT, tauR (constantes de temps poussée / couples), delay (s), com(2)
THETA_NOM = dict(m=1.0, J=np.array([0.005, 0.005, 0.009]), Drot=np.array([0.015, 0.015, 0.020]),
                 cd=0.10, cl=0.05, kw=0.08, lag=True, tauT=0.05, tauR=0.02, delay=0.02,
                 com=np.zeros(2))
THETA_IDEAL = dict(THETA_NOM, cd=0.0, cl=0.0, lag=False, delay=0.0)   # plante = modèle physique nominal
THETA_MODEL = dict(THETA_IDEAL, kw=0.0)                                # ce que « sait » le MPC physique

def plant_rhs(x, Ta, tau, th, m, vw):
    p, v, Rm, w = state_unpack(x)
    F = (-th['cd']*np.linalg.norm(v)*v - th['cl']*v
         + th['kw']*np.linalg.norm(vw)*vw)
    vdot = (Rm @ np.array([0., 0., Ta]) + F)/m - np.array([0., 0., G])
    ct = np.array([th['com'][1]*Ta, -th['com'][0]*Ta, 0.])
    J = th['J']
    wdot = (tau + ct - np.cross(w, J*w) - th['Drot']*w)/J
    return np.r_[v, vdot, (Rm @ hat(w)).reshape(-1), wdot]

def rk4_raw(x, Ta, tau, th, m, vw, h):
    k1 = plant_rhs(x, Ta, tau, th, m, vw)
    k2 = plant_rhs(x+0.5*h*k1, Ta, tau, th, m, vw)
    k3 = plant_rhs(x+0.5*h*k2, Ta, tau, th, m, vw)
    k4 = plant_rhs(x+h*k3, Ta, tau, th, m, vw)
    return x + h*(k1+2*k2+2*k3+k4)/6.0

def wind_series(dist, n, rng):
    """Vitesse du vent (n,3) à la cadence DT_SIM : rampe + rafales OU + sinus lent."""
    vw = np.zeros((n, 3))
    if not dist.get('wind'): return vw
    t = np.arange(n)*DT_SIM
    t0, t1 = dist['t0'], dist['t1']
    ramp = np.clip(np.minimum((t-t0)/1.0, (t1-t)/1.0), 0.0, 1.0)
    vw += ramp[:, None]*dist['Wvec'][None, :]
    a = math.exp(-DT_SIM/dist['tau_c'])
    g = np.zeros((n, 3)); xk = np.zeros(3)
    xi = rng.standard_normal((n, 3))
    for k in range(n):
        xk = a*xk + dist['sigma']*math.sqrt(1-a*a)*xi[k]; g[k] = xk
    vw += ramp[:, None]*g
    vw[:, :2] += (ramp*dist['sin_amp']*np.sin(2*np.pi*dist['sin_f']*t + dist['sin_ph']))[:, None]
    return vw

class Plant:
    def __init__(self, spec):
        th = spec['theta']; self.th = th
        n = int(spec['T']/DT_SIM) + SUBSTEPS + 2
        self.vw = wind_series(spec['dist'], n, np.random.default_rng(spec['wind_seed']))
        pl = spec['dist'].get('payload')
        self.pl_ratio, self.pl_t = (pl['ratio'], pl['t']) if pl else (1.0, 1e9)
        K = int(round(th['delay']/DT_SIM))
        self.K = K; self.buf = [U_HOVER.copy() for _ in range(K)]
        self.Ta = U_HOVER[0]; self.tau = U_HOVER[1:].copy()
        self.aT = 1-math.exp(-DT_SIM/th['tauT']) if th['lag'] else 1.0
        self.aR = 1-math.exp(-DT_SIM/th['tauR']) if th['lag'] else 1.0
        self.k = 0

    def step(self, x, u):
        th = self.th
        for _ in range(SUBSTEPS):
            t = self.k*DT_SIM
            if self.K == 0: ue = u
            else: self.buf.append(u); ue = self.buf.pop(0)
            ue = np.clip(ue, U_MIN, U_MAX)
            self.Ta += self.aT*(ue[0]-self.Ta); self.tau += self.aR*(ue[1:]-self.tau)
            m = th['m']*(self.pl_ratio if t >= self.pl_t else 1.0)
            x = rk4_raw(x, self.Ta, self.tau, th, m, self.vw[self.k], DT_SIM)
            p, v, Rm, w = state_unpack(x); x = state_pack(p, v, project_so3(Rm), w)
            self.k += 1
        return x

SPLIT_ID = {'id': 0, 'tune': 1, 'val': 2, 'test': 3, 'abl': 4}
REGIME_ID = {'mismatch': 0, 'ideal': 1}
KINDS = ['nominal', 'wind', 'payload', 'combined']
FAMS = ['circle', 'figure8', 'agile']

def sample_plant(rng, regime):
    if regime == 'ideal': return dict(THETA_IDEAL)
    th = dict(THETA_NOM)
    th['m'] = THETA_NOM['m']*rng.uniform(0.92, 1.08)
    th['J'] = THETA_NOM['J']*rng.uniform(0.85, 1.20, 3)
    th['Drot'] = THETA_NOM['Drot']*rng.uniform(0.7, 1.3, 3)
    th['cd'] = THETA_NOM['cd']*rng.uniform(0.5, 1.5); th['cl'] = THETA_NOM['cl']*rng.uniform(0.5, 1.5)
    th['tauT'] = rng.uniform(0.03, 0.08); th['tauR'] = rng.uniform(0.01, 0.035)
    th['delay'] = float(rng.choice([0.0, 0.01, 0.02, 0.03]))
    th['com'] = rng.uniform(-0.008, 0.008, 2)
    return th

def sample_dist(rng, kind, T, strength=1.0):
    d = {'wind': kind in ('wind', 'combined'), 'payload': None}
    if d['wind']:
        ang = rng.uniform(0, 2*np.pi); W0 = rng.uniform(1.0, 5.0)*strength
        d.update(t0=rng.uniform(0.15, 0.35)*T, Wvec=np.array([W0*np.cos(ang), W0*np.sin(ang), rng.uniform(-0.3, 0.3)*W0]),
                 sigma=rng.uniform(0.3, 1.2)*strength, tau_c=rng.uniform(0.8, 2.0),
                 sin_amp=rng.uniform(0.0, 1.0)*strength, sin_f=rng.uniform(0.05, 0.25), sin_ph=rng.uniform(0, 2*np.pi))
        d['t1'] = d['t0'] + rng.uniform(0.2, 0.35)*T
    if kind in ('payload', 'combined'):
        d['payload'] = dict(ratio=rng.uniform(0.8, 1.3), t=rng.uniform(0.3, 0.6)*T)
    return d

def make_trial(split, i, regime='mismatch', kind=None, T=None):
    T = T or T_DUR
    rng = np.random.default_rng([SEED, SPLIT_ID[split], int(i), REGIME_ID[regime]])
    kind = kind or KINDS[i % 4]; fam = FAMS[(i//4) % 3]
    return dict(split=split, trial=int(i), regime=regime, kind=kind, family=fam, T=T,
                theta=sample_plant(rng, regime), dist=sample_dist(rng, kind, T),
                wind_seed=int(rng.integers(1 << 31)), meas_seed=int(rng.integers(1 << 31)))

def initial_state(rng):
    return state_pack(np.array([0., 0., 1.5]) + rng.uniform(-0.1, 0.1, 3), rng.uniform(-0.05, 0.05, 3),
                      np.eye(3), rng.uniform(-0.03, 0.03, 3))

def noisy_state(x, rng, s=1.0):
    p, v, Rm, w = state_unpack(x)
    p = p + rng.normal(0, 0.005*s, 3); v = v + rng.normal(0, 0.01*s, 3); w = w + rng.normal(0, 0.01*s, 3)
    Rm = project_so3(Rm @ expm_so3(np.deg2rad(0.2*s)*rng.normal(size=3)))
    return state_pack(p, v, Rm, w)

In [ ]:
# =============================================================================
# 3. CONTRÔLEURS : DOB, PID+DOB, MPC linéaire (modèle physique OU Koopman)
# =============================================================================
class DisturbanceObserver:
    """Observateur d'accélération perturbatrice (Luenberger étendu, discret, 2 paramètres).
    v_hat+ = v_hat + dt (a_nom + d_hat + L1 e) ; d_hat+ = d_hat + dt L2 e ; L1=2w0, L2=w0^2."""
    def __init__(self, omega=5.0, dt=DT, d_max=DOB_D_MAX):
        self.L1, self.L2, self.dt, self.d_max = 2.0*omega, omega**2, dt, d_max
        self.v_hat = None; self.d_hat = np.zeros(3); self._e = np.zeros(3)
    def correct(self, v):
        if self.v_hat is None: self.v_hat = np.asarray(v, float).copy()
        self._e = v - self.v_hat
        self.d_hat = np.clip(self.d_hat + self.dt*self.L2*self._e, -self.d_max, self.d_max)
        return self.d_hat
    def predict(self, a_nom):
        self.v_hat = self.v_hat + self.dt*(a_nom + self.d_hat + self.L1*self._e)

# Gains d'attitude pour une commande à 20 Hz (DT=0.05) et J=0.005 : omega_n ~ 12 rad/s, zeta ~ 0.9
#   kR = J*omega_n^2 ~ 0.7 ; kw = 2*zeta*omega_n*J ~ 0.10.  (Les gains hérités kR=120, kw=15 donnent
#   dt*kw/J = 150 >> 2 : boucle bang-bang permanente sur +-TAU_MAX, non représentative d'une base « sérieuse ».)
PID_DEFAULT = dict(kp_xy=12., kd_xy=6., kp_z=16., kd_z=8., kR=0.7, kw=0.10, w_alpha=1.0, omega=5.0)

class GeometricPIDDOB:
    """PID géométrique (Lee 2010), sans intégrale, + DOB. use_dob=False => PD géométrique seul
    (utilisé uniquement pour générer les données d'identification)."""
    def __init__(self, params=None, use_dob=True):
        p = dict(PID_DEFAULT); p.update(params or {}); self.p = p; self.use_dob = use_dob
        self.obs = DisturbanceObserver(p['omega']); self.w_f = None
        self.kp = np.array([p['kp_xy'], p['kp_xy'], p['kp_z']]); self.kd = np.array([p['kd_xy'], p['kd_xy'], p['kd_z']])
    def __call__(self, x, t, family='circle'):
        p_, v, Rm, w = state_unpack(x)
        pref, vref, aref = reference(t, family)
        d_hat = self.obs.correct(v) if self.use_dob else np.zeros(3)
        self.w_f = w.copy() if self.w_f is None else self.w_f + self.p['w_alpha']*(w - self.w_f)
        F = aref + self.kp*(pref-p_) + self.kd*(vref-v) - d_hat + np.array([0, 0, G])
        Tc = np.linalg.norm(F); T = float(np.clip(Tc, 0.0, T_MAX)); b3d = F/(Tc+1e-9)
        b1d = np.array([1., 0., 0.]); b2d = np.cross(b3d, b1d)
        if np.linalg.norm(b2d) < 1e-5: b1d = np.array([0., 1., 0.]); b2d = np.cross(b3d, b1d)
        b2d /= np.linalg.norm(b2d); b1d = np.cross(b2d, b3d)
        Rd = np.column_stack([b1d, b2d, b3d])
        Re = 0.5*(Rd.T@Rm - Rm.T@Rd); eR = np.array([Re[2, 1], Re[0, 2], Re[1, 0]])
        tau = np.clip(-self.p['kR']*eR - self.p['kw']*self.w_f, -TAU_MAX, TAU_MAX)
        if self.use_dob: self.obs.predict(Rm[:, 2]*T - np.array([0., 0., G]))
        return np.r_[T, tau]

# ---------- Dictionnaires Koopman ----------
VARIANT_FLAGS = {'lin': (0, 0), 'drag': (1, 0), 'lag': (0, 1), 'full': (1, 1)}   # (traînée ‖v‖v, retards d'entrée)

def lift(x, u1, u2, variant, aux=None):
    if variant == 'x': return x.copy()
    if variant == 'physlag': return np.concatenate([x, aux])      # x + état d'actionneur estimé
    dr, lg = VARIANT_FLAGS[variant]
    parts = [x]
    if dr: v = x[3:6]; parts.append(np.linalg.norm(v)*v)
    if lg: parts += [u1, u2]
    return np.concatenate(parts)

def fit_koopman(data, variant, lam):
    """EDMD ridge standardisé avec entrées. Les lignes des retards d'entrée sont exactes (u1+=u, u2+=u1)."""
    X, U1, U2, U, Xn = data
    dr, lg = VARIANT_FLAGS[variant]
    Z  = np.array([lift(x, a, b, variant) for x, a, b in zip(X, U1, U2)])
    Zn = np.array([lift(x, a, b, variant) for x, a, b in zip(Xn, U, U1)])
    nz = Z.shape[1]; nfit = 18 + 3*dr
    Phi = np.column_stack([Z, U])
    mu = Phi.mean(0); sig = np.where(Phi.std(0) < 1e-8, 1.0, Phi.std(0))
    Ps = np.column_stack([(Phi-mu)/sig, np.ones(len(Phi))])
    Th = (Zn[:, :nfit].T @ Ps) @ np.linalg.inv(Ps.T@Ps + lam*np.eye(Ps.shape[1]))
    W = Th[:, :-1]/sig; c_fit = Th[:, -1] - W @ mu
    A = np.zeros((nz, nz)); B = np.zeros((nz, 4)); c = np.zeros(nz)
    A[:nfit] = W[:, :nz]; B[:nfit] = W[:, nz:]; c[:nfit] = c_fit
    if lg:
        B[nfit:nfit+4] = np.eye(4); A[nfit+4:nfit+8, nfit:nfit+4] = np.eye(4)
    return dict(A=A, B=B, c=c, nz=nz, variant=variant, lam=lam)

def stabilize_spectrum(model, rho_max=RHO_MAX):
    """Projette le rayon spectral de A sur rho_max via une décomposition de Schur RÉELLE (base
    orthogonale Z, A = Z T Z^T) : chaque bloc diagonal de T (1x1 réel, ou 2x2 = paire complexe conjuguée)
    dont le module dépasse rho_max est ramené à ce module par simple mise à l'échelle du bloc, les autres
    sont inchangés, puis A_stab = Z T' Z^T. Contrairement à une décomposition en valeurs/vecteurs propres
    (numpy.linalg.eig), la base de Schur est TOUJOURS orthogonale — donc numériquement stable même quand A
    est quasi défective (ce qui est le cas ici : le bloc de mémoire d'entrée du dictionnaire Koopman rend
    les vecteurs propres de eig() presque colinéaires, cond(V) ~ 1e32, et une reconstruction par inversion
    de V fait exploser les coefficients de A_stab à ~1e274 — testé et rejeté). B et c ne sont PAS modifiés."""
    A = model['A']; rho_before = float(np.max(np.abs(np.linalg.eigvals(A))))
    T, Z = schur(A, output='real'); Tc = T.copy(); n = T.shape[0]; n_clipped = 0; i = 0
    while i < n:
        if i+1 < n and abs(T[i+1, i]) > 1e-10:                         # bloc complexe 2x2
            blk = Tc[i:i+2, i:i+2]; det = blk[0, 0]*blk[1, 1] - blk[0, 1]*blk[1, 0]
            modulus = math.sqrt(max(det, 0.0))
            if modulus > rho_max: Tc[i:i+2, i:i+2] = blk*(rho_max/modulus); n_clipped += 2
            i += 2
        else:                                                          # bloc réel 1x1
            if abs(Tc[i, i]) > rho_max: Tc[i, i] = math.copysign(rho_max, Tc[i, i]); n_clipped += 1
            i += 1
    A_stab = Z @ Tc @ Z.T
    rho_after = float(np.max(np.abs(np.linalg.eigvals(A_stab))))
    m2 = dict(model); m2['A'] = A_stab; m2['variant'] = model['variant']
    m2['rho_before'], m2['rho_after'], m2['n_clipped'] = rho_before, rho_after, n_clipped
    if 'act_alpha' in model: m2['act_alpha'] = model['act_alpha']
    return m2

def ref_lift(model, t, family):
    """État de référence dans les coordonnées du modèle (koopman ou physique) : valeurs de la trajectoire
    désirée sur les lignes physiques (p, v, attitude = niveau, omega = 0), valeurs neutres ailleurs
    (feature de traînée évaluée à vref, mémoire d'entrée = hover). Ce n'est PAS un point d'équilibre exact
    du modèle — la trajectoire n'en est pas un — c'est une cible pour la commande proportionnelle d'état,
    au même titre que Rd dans le PID géométrique."""
    pref, vref, _ = reference(t, family)
    x_ref = state_pack(pref, vref, np.eye(3), np.zeros(3))
    variant = model['variant']
    if variant == 'physlag': return np.r_[x_ref, U_HOVER]
    if variant == 'x': return x_ref
    dr, lg = VARIANT_FLAGS[variant]; parts = [x_ref]
    if dr: parts.append(np.linalg.norm(vref)*vref)
    if lg: parts += [U_HOVER, U_HOVER]
    return np.concatenate(parts)

def physics_model(with_lag=False):
    """Linéarisation numérique (hover) de la carte 1-pas NOMINALE : m=1, J, Drot nominaux ; sans traînée,
    sans vent, sans retard. with_lag=False : ni lag actionneur (modèle « naïf »).
    with_lag=True : base « gray-box » = + lag moteur au temps de réponse NOMINAL (tauT, tauR de THETA_NOM),
    état d'actionneur estimé par récurrence sur les commandes appliquées. Traînée, paramètres réels et retard
    restent inconnus du modèle."""
    th = THETA_MODEL
    aT = 1-math.exp(-DT_SIM/THETA_NOM['tauT']); aR = 1-math.exp(-DT_SIM/THETA_NOM['tauR'])
    al = np.array([aT, aR, aR, aR]); nx = 22 if with_lag else 18
    def f(z, u):
        xs = z[:18].copy(); act = z[18:22].copy() if with_lag else None
        for _ in range(SUBSTEPS):
            if with_lag: act = act + al*(u-act); xs = rk4_raw(xs, act[0], act[1:], th, 1.0, np.zeros(3), DT_SIM)
            else: xs = rk4_raw(xs, u[0], u[1:], th, 1.0, np.zeros(3), DT_SIM)
        return np.r_[xs, act] if with_lag else xs
    x0 = state_pack(np.array([0., 0., 1.5]), np.zeros(3), np.eye(3), np.zeros(3)); u0 = U_HOVER.copy(); e = 1e-6
    z0 = np.r_[x0, u0] if with_lag else x0; I = np.eye(nx)
    A = np.column_stack([(f(z0+e*I[i], u0) - f(z0-e*I[i], u0))/(2*e) for i in range(nx)])
    B = np.column_stack([(f(z0, u0+e*np.eye(4)[i]) - f(z0, u0-e*np.eye(4)[i]))/(2*e) for i in range(4)])
    c = f(z0, u0) - A@z0 - B@u0
    m = dict(A=A, B=B, c=c, nz=nx, variant=('physlag' if with_lag else 'x'), lam=0.0)
    if with_lag: m['act_alpha'] = 1-np.exp(-DT/np.array([THETA_NOM['tauT']]+[THETA_NOM['tauR']]*3))
    return m

# ---------- Cœur MPC condensé (matrices constantes précalculées) ----------
class MPCCore:
    """QP condensé  min_U  1/2 U'HU + q'U  avec H constant ; q affine en (z0, c_eff, yref, u_prev).
    Toute la partie coûteuse est précalculée : un pas MPC = quelques produits matrice-vecteur + OSQP.update(q)."""
    def __init__(self, model, qscale=1.0, rscale=1.0, N=N_H):
        A, B, c = model['A'], model['B'], model['c']; nz = A.shape[0]; nu = 4
        self.model, self.N, self.nz, self.qscale = model, N, nz, qscale
        F = np.zeros((N, nz, nz)); S = np.zeros((N, nz, N, nu)); Gc = np.zeros((N, nz, nz))
        Fp = np.eye(nz); Gp = np.zeros((nz, nz)); Sp = np.zeros((nz, N, nu))
        for j in range(N):
            Fp = A@Fp; Gp = A@Gp + np.eye(nz); Sp = np.einsum('ab,bkl->akl', A, Sp); Sp[:, j, :] = B
            F[j], Gc[j], S[j] = Fp, Gp, Sp
        Cy = np.zeros((12, nz)); Cy[0:3, 0:3] = np.eye(3); Cy[3:6, 3:6] = np.eye(3)
        Cy[6:9, [8, 11, 14]] = np.eye(3); Cy[9:12, 15:18] = np.eye(3)
        Sy = np.einsum('ra,jakl->jrkl', Cy, S).reshape(N*12, N*nu)
        Fy = np.einsum('ra,jab->jrb', Cy, F).reshape(N*12, nz)
        Gy = np.einsum('ra,jab->jrb', Cy, Gc).reshape(N*12, nz)
        Qy = block_diag(Q*qscale, Qv*qscale, Qt, Qw)
        Qb = [Qy.copy() for _ in range(N)]; Qb[-1][:3, :3] += QN_FACTOR*Q*qscale
        Qb = block_diag(*Qb)
        D = np.eye(N*nu) - np.eye(N*nu, k=-nu)
        H = 2*Sy.T@Qb@Sy + 2*rscale*np.kron(np.eye(N), R_U) + 2*rscale*D.T@np.kron(np.eye(N), RD_U)@D
        self.P = sparse.csc_matrix((H+H.T)/2 + 1e-8*np.eye(N*nu))
        self.KZ, self.KC, self.KY = 2*Sy.T@Qb@Fy, 2*Sy.T@Qb@Gy, 2*Sy.T@Qb
        self.q_u = np.tile(-2*rscale*R_U@U_HOVER, N); self.rscale = rscale
        self.Acon = sparse.eye(N*nu, format='csc')
        self.lo, self.hi = np.tile(U_MIN, N), np.tile(U_MAX, N)
        self.Bd = np.zeros((nz, 3)); self.Bd[0:3] = 0.5*DT**2*np.eye(3); self.Bd[3:6] = DT*np.eye(3)

_CORE_CACHE = {}
def get_core(ctx, model_key, qscale, rscale=1.0):
    key = (ctx['regime'], model_key, float(qscale), float(rscale))
    if key not in _CORE_CACHE: _CORE_CACHE[key] = MPCCore(ctx['models'][model_key], qscale, rscale)
    return _CORE_CACHE[key]

class LinearMPC:
    """MPC linéaire sur modèle (physique ou Koopman). dob=True : d_hat décale c (c_eff = c + Bd d_hat)."""
    def __init__(self, core, dob=True, omega=5.0):
        self.core, self.dob = core, dob; self.obs = DisturbanceObserver(omega)
        self.u1, self.u2 = U_HOVER.copy(), U_HOVER.copy(); self.aux = U_HOVER.copy(); self.n_fail = 0
        self.solver = osqp.OSQP()
        self.solver.setup(P=core.P, q=np.zeros(core.N*4), A=core.Acon, l=core.lo, u=core.hi, verbose=False,
                          eps_abs=1e-5, eps_rel=1e-5, max_iter=2000, polishing=False, warm_start=True)
    def __call__(self, x, t, family='circle'):
        co = self.core; m = co.model; N = co.N
        z0 = lift(x, self.u1, self.u2, m['variant'], self.aux)
        d_hat = self.obs.correct(x[3:6]) if self.dob else np.zeros(3)
        c_eff = m['c'] + co.Bd@d_hat
        refs = [reference(t+(j+1)*DT, family) for j in range(N)]
        yref = np.concatenate([np.r_[r[0], r[1], [0., 0., 1.], np.zeros(3)] for r in refs])
        q = co.KZ@z0 + co.KC@c_eff - co.KY@yref + co.q_u
        q[:4] += -2*co.rscale*RD_U@self.u1
        self.solver.update(q=q); res = self.solver.solve()
        ok = res.x is not None and 'solved' in str(res.info.status).lower()
        if ok: u = np.clip(res.x[:4], U_MIN, U_MAX)
        else: u = U_HOVER.copy(); self.n_fail += 1
        if self.dob:
            zn = m['A']@z0 + m['B']@u + m['c']
            self.obs.predict((zn[3:6] - z0[3:6])/DT)
        if m['variant'] == 'physlag': self.aux = self.aux + m['act_alpha']*(u - self.aux)
        self.u2, self.u1 = self.u1, u
        return u

def lqr_gain(model, qscale=1.0, rscale=1.0):
    """Gain de retour d'état LQR à horizon infini (Riccati discrète) sur le modèle donné, avec les MÊMES
    poids de sortie (Q, Qv, Qt, Qw) et de commande (R_U) que le MPC — seule la pondération du taux de
    variation de commande (RD_U) n'a pas d'équivalent standard en LQR à horizon infini sans augmenter
    l'état ; elle est omise ici (limite à signaler)."""
    A, B = model['A'], model['B']; nz = A.shape[0]
    Cy = np.zeros((12, nz)); Cy[0:3, 0:3] = np.eye(3); Cy[3:6, 3:6] = np.eye(3)
    Cy[6:9, [8, 11, 14]] = np.eye(3); Cy[9:12, 15:18] = np.eye(3)
    Qy = block_diag(Q*qscale, Qv*qscale, Qt, Qw)
    Qfull = Cy.T@Qy@Cy + 1e-9*np.eye(nz); Rfull = rscale*R_U
    P = solve_discrete_are(A, B, Qfull, Rfull)
    K = np.linalg.solve(Rfull + B.T@P@B, B.T@P@A)
    return K

_LQR_CACHE = {}
def get_lqr_gain(ctx, model_key, qscale, rscale=1.0):
    key = (ctx['regime'], model_key, float(qscale), float(rscale))
    if key not in _LQR_CACHE: _LQR_CACHE[key] = lqr_gain(ctx['models'][model_key], qscale, rscale)
    return _LQR_CACHE[key]

class KoopmanLQR:
    """Retour d'état LQR (gain fixe, horizon infini) + anticipation 1 pas (inversion de B pour rejoindre
    z_ref(t+dt) depuis z_ref(t)) + DOB offset-free (même mécanisme que LinearMPC : d_hat décale c). Comparé
    au MPC sur le MÊME modèle, seule la stratégie de commande change (gain fixe + anticipation locale, vs
    optimisation sur un horizon glissant avec contraintes) : ablation MPC vs LQR à modèle égal."""
    def __init__(self, model, K, omega=5.0):
        self.model, self.K = model, K; nz = model['A'].shape[0]
        self.Bpinv = np.linalg.pinv(model['B']); self.obs = DisturbanceObserver(omega)
        self.Bd = np.zeros((nz, 3)); self.Bd[0:3] = 0.5*DT**2*np.eye(3); self.Bd[3:6] = DT*np.eye(3)
        self.u1, self.u2 = U_HOVER.copy(), U_HOVER.copy(); self.aux = U_HOVER.copy(); self.n_fail = 0
    def __call__(self, x, t, family='circle'):
        m = self.model
        z = lift(x, self.u1, self.u2, m['variant'], self.aux)
        d_hat = self.obs.correct(x[3:6]); c_eff = m['c'] + self.Bd@d_hat
        zr0, zr1 = ref_lift(m, t, family), ref_lift(m, t+DT, family)
        u_ff = self.Bpinv @ (zr1 - m['A']@zr0 - c_eff)
        u_raw = u_ff - self.K@(z - zr0)
        u = np.clip(u_raw, U_MIN, U_MAX)
        if np.any(u != u_raw): self.n_fail += 1     # saturation actionneur (pas d'échec QP : compté à part)
        zn = m['A']@z + m['B']@u + m['c']; self.obs.predict((zn[3:6]-z[3:6])/DT)
        if m['variant'] == 'physlag': self.aux = self.aux + m['act_alpha']*(u - self.aux)
        self.u2, self.u1 = self.u1, u
        return u

def make_controller(name, ctx, override=None):
    ov = override or {}
    if name == 'pid_dob':
        return GeometricPIDDOB(ov.get('pid_params', ctx['pid_params']))
    if name == 'pid_default':
        return GeometricPIDDOB(None)
    if name == 'pid_plain':
        return GeometricPIDDOB(None, use_dob=False)
    if name.startswith('lqr_'):
        model_key = name[4:-4] if name.endswith('_dob') else name[4:]     # ex: 'lqr_koopstab_dob' -> 'koopstab'
        hp = ctx['mpc'].get(f'lqr_{model_key}', {})
        qs = ov.get('qscale', hp.get('qscale', 1.0)); rs = ov.get('rscale', hp.get('rscale', 1.0)); om = ov.get('omega', hp.get('omega', 5.0))
        K = get_lqr_gain(ctx, model_key, qs, rs)
        return KoopmanLQR(ctx['models'][model_key], K, omega=om)
    dob = name.endswith('_dob'); base = name[:-4] if dob else name
    assert base.endswith('_mpc'), name
    model_key = base[:-4]
    hp = ctx['mpc'].get(model_key, {})
    qs = ov.get('qscale', hp.get('qscale', 1.0)); rs = ov.get('rscale', hp.get('rscale', 1.0)); om = ov.get('omega', hp.get('omega', 5.0))
    return LinearMPC(get_core(ctx, model_key, qs, rs), dob=dob, omega=om)

CTRL_LABEL = {'pid_dob': 'PID+DOB (tuned)', 'physlag_mpc_dob': 'Gray-box MPC+DOB', 'koop_mpc_dob': 'Koopman MPC+DOB',
              'koopstab_mpc_dob': 'Koopman-stable MPC+DOB', 'lqr_koopstab_dob': 'Koopman-stable+LQR+DOB',
              'lqr_koop_dob': 'Koopman(brut)+LQR+DOB',
              'phys_mpc_dob': 'Naive physics MPC+DOB', 'koop_mpc': 'Koopman MPC (no DOB)',
              'physlag_mpc': 'Gray-box MPC (no DOB)', 'koopstab_mpc': 'Koopman-stable MPC (no DOB)',
              'koop-lin_mpc_dob': 'Koopman-lin+DOB', 'koop-drag_mpc_dob': 'Koopman-drag+DOB',
              'koop-lag_mpc_dob': 'Koopman-lag+DOB', 'koop-full_mpc_dob': 'Koopman-full+DOB'}
MAIN_CTRLS = ['pid_dob', 'physlag_mpc_dob', 'koop_mpc_dob', 'koopstab_mpc_dob', 'lqr_koopstab_dob', 'lqr_koop_dob']

In [ ]:
# =============================================================================
# 4. SIMULATION EN BOUCLE FERMÉE + MOTEUR DE JOBS PARALLÈLE (avec checkpoint)
# =============================================================================
def run_trial(ctrl, spec, ctx, noise_scale=1.0, override=None, keep_log=False):
    n = int(spec['T']/DT); fam = spec['family']
    rng = np.random.default_rng(spec['meas_seed']); plant = Plant(spec)
    controller = make_controller(ctrl, ctx, override)
    x = initial_state(rng); pref0, vref0, _ = reference(0.0, fam)
    x[:3] = pref0 + rng.uniform(-0.05, 0.05, 3); x[3:6] = vref0
    P, PR, U, CPU, X = [], [], [], [], []
    fail, t_fail = False, np.nan
    for k in range(n):
        t = k*DT; xm = noisy_state(x, rng, noise_scale)
        tic = time.perf_counter(); u = controller(xm, t, fam); CPU.append((time.perf_counter()-tic)*1e3)
        pref = reference(t, fam)[0]
        P.append(x[:3].copy()); PR.append(pref); U.append(u.copy()); X.append(x.copy())
        x = plant.step(x, u)
        if (not np.all(np.isfinite(x))) or np.linalg.norm(x[:3]-reference(t+DT, fam)[0]) > FAIL_ERR \
                or np.linalg.norm(x[15:18]) > FAIL_W:
            fail, t_fail = True, t+DT; break
    P, PR, U = np.array(P), np.array(PR), np.array(U); en = np.linalg.norm(P-PR, axis=1)
    rmse = float(np.sqrt(np.mean(en**2)))
    m = dict(fail=int(fail), t_fail=t_fail,
             RMSE=(np.nan if fail else rmse), RMSE_pen=(RMSE_CAP if fail else min(rmse, RMSE_CAP)),
             MAE=float(np.mean(en)), P95Error=float(np.percentile(en, 95)), MaxError=float(np.max(en)),
             FinalError=float(en[-1]),
             ControlVariation=float(np.sum(np.sum(np.diff(U, axis=0)**2, axis=1))*DT) if len(U) > 1 else 0.0,
             CPU_mean_ms=float(np.mean(CPU)), CPU_max_ms=float(np.max(CPU)),
             qp_fail=int(getattr(controller, 'n_fail', 0)))
    if keep_log: m['log'] = dict(t=np.arange(len(P))*DT, p=P, pref=PR, u=U, x=np.array(X))
    return m

CTX_ALL = {}
def _worker(job):
    ctx = CTX_ALL[job['regime']]
    spec = make_trial(job['split'], job['trial'], job['regime'], kind=job.get('kind'))
    m = run_trial(job['ctrl'], spec, ctx, noise_scale=job.get('noise', 1.0), override=job.get('override'))
    return dict(key=job['key'], regime=job['regime'], split=job['split'], trial=job['trial'],
                kind=spec['kind'], family=spec['family'], ctrl=job['ctrl'], tag=job.get('tag', ''), **m)

def run_jobs(jobs, ckpt=None, sig='', desc='jobs'):
    """Exécute les jobs (parallèle si N_JOBS>1). Si ckpt : reprise via fichier JSONL + signature."""
    done, rows = set(), []
    if ckpt is not None:
        ckpt = Path(ckpt); sigf = ckpt.with_suffix('.sig')
        if ckpt.exists() and sigf.exists() and sigf.read_text() == sig:
            for line in ckpt.read_text().splitlines():
                r = json.loads(line); rows.append(r); done.add(r['key'])
        else:
            if ckpt.exists(): ckpt.rename(ckpt.with_suffix('.jsonl.bak'))
            sigf.write_text(sig)
    todo = [j for j in jobs if j['key'] not in done]
    print(f"    [{desc}] {len(jobs)} jobs, {len(done)} déjà faits, {len(todo)} à faire (N_JOBS={N_JOBS})", flush=True)
    t0 = time.time(); fh = open(ckpt, 'a') if ckpt is not None else None
    def sink(r):
        rows.append(r)
        if fh is not None: fh.write(json.dumps(r, default=float)+'\n'); fh.flush()
    try:
        if N_JOBS > 1 and len(todo) > 1:
            with mp.get_context('fork').Pool(N_JOBS) as pool:
                for i, r in enumerate(pool.imap_unordered(_worker, todo, chunksize=1)):
                    sink(r)
                    if (i+1) % 25 == 0: print(f"      {i+1}/{len(todo)}  ({time.time()-t0:.0f}s)", flush=True)
        else:
            for i, j in enumerate(todo):
                sink(_worker(j))
                if (i+1) % 25 == 0: print(f"      {i+1}/{len(todo)}  ({time.time()-t0:.0f}s)", flush=True)
    finally:
        if fh is not None: fh.close()
    return pd.DataFrame(rows)

In [ ]:
# =============================================================================
# 5. IDENTIFICATION, VALIDATION BOUCLE OUVERTE, SÉLECTION DU DICTIONNAIRE
# =============================================================================
def collect_id_data(regime, n_ep=N_EP_ID, n_steps=N_STEPS_ID):
    """Données d'identification sur la plante NOMINALE du régime (paramètres vus à l'id),
    excitation : PD géométrique + dither large ; la moitié des épisodes avec rafales/vent.
    États non bruités (hypothèse : états lissés hors ligne)."""
    rng = np.random.default_rng([SEED, SPLIT_ID['id'], REGIME_ID[regime]])
    th_id = dict(THETA_NOM if regime == 'mismatch' else THETA_IDEAL)
    X, U1, U2, U, Xn = [], [], [], [], []
    for ep in range(n_ep):
        fam = FAMS[ep % 3]; kind = 'wind' if ep % 2 else 'nominal'
        dist = sample_dist(rng, kind, n_steps*DT, strength=0.6)
        spec = dict(theta=th_id, T=n_steps*DT, dist=dist, wind_seed=int(rng.integers(1 << 31)))
        plant = Plant(spec); pid = GeometricPIDDOB(None, use_dob=False)
        x = initial_state(rng); x[:3] = reference(0, fam)[0] + rng.uniform(-0.05, 0.05, 3); x[3:6] = reference(0, fam)[1]
        u1, u2 = U_HOVER.copy(), U_HOVER.copy()
        for k in range(n_steps):
            u = np.clip(pid(x, k*DT, fam) + np.r_[rng.uniform(-3, 3), rng.uniform(-0.05, 0.05, 3)], U_MIN, U_MAX)
            xn = plant.step(x, u)
            X.append(x); U1.append(u1); U2.append(u2); U.append(u); Xn.append(xn)
            x, u2, u1 = xn, u1, u
    return tuple(np.asarray(a) for a in (X, U1, U2, U, Xn))

def collect_val_trajs(regime, n_ep=N_VAL, n_steps=120):
    """Trajectoires de validation sur plantes HORS identification (params tirés), sans vent : erreur de modèle pure."""
    out = []
    for i in range(n_ep):
        spec = make_trial('val', i, regime, kind='nominal', T=n_steps*DT)
        rng = np.random.default_rng(spec['meas_seed']); plant = Plant(spec); pid = GeometricPIDDOB(None, use_dob=False)
        fam = spec['family']; x = initial_state(rng); x[:3] = reference(0, fam)[0]; x[3:6] = reference(0, fam)[1]
        Xs, Us = [x.copy()], []
        for k in range(n_steps):
            u = np.clip(pid(x, k*DT, fam) + np.r_[rng.uniform(-1.5, 1.5), rng.uniform(-0.03, 0.03, 3)], U_MIN, U_MAX)
            x = plant.step(x, u); Us.append(u); Xs.append(x.copy())
        out.append((np.array(Xs), np.array(Us)))
    return out

HORIZONS = [1, 5, 10, 20]
def open_loop_errors(model, val):
    errp = {h: [] for h in HORIZONS}; errv = {h: [] for h in HORIZONS}; Hm = max(HORIZONS)
    A, B, c = model['A'], model['B'], model['c']
    for Xs, Us in val:
        aux = [U_HOVER.copy()]
        if model['variant'] == 'physlag':
            for u in Us: aux.append(aux[-1] + model['act_alpha']*(u - aux[-1]))
        for k in range(2, len(Us)-Hm):
            z = lift(Xs[k], Us[k-1], Us[k-2], model['variant'], aux[k] if model['variant'] == 'physlag' else None)
            for j in range(Hm):
                z = A@z + B@Us[k+j] + c
                if (j+1) in errp:
                    errp[j+1].append(np.linalg.norm(z[:3]-Xs[k+j+1, :3])); errv[j+1].append(np.linalg.norm(z[3:6]-Xs[k+j+1, 3:6]))
    return {h: (float(np.sqrt(np.mean(np.square(errp[h])))), float(np.sqrt(np.mean(np.square(errv[h]))))) for h in HORIZONS}

def identify_regime(regime):
    print(f"\n[ID:{regime}] collecte ({N_EP_ID} épisodes)…", flush=True)
    data = collect_id_data(regime); val = collect_val_trajs(regime)
    models, rows = {'phys': physics_model(False), 'physlag': physics_model(True)}, []
    for mk in ('phys', 'physlag'):
        for h, (ep_, ev_) in open_loop_errors(models[mk], val).items():
            rows.append(dict(regime=regime, model=mk, lam=np.nan, horizon=h, pos_rmse=ep_, vel_rmse=ev_))
    best = {}
    for variant in VARIANT_FLAGS:
        for lam in (1e-6, 1e-4, 1e-2):
            m = fit_koopman(data, variant, lam); e = open_loop_errors(m, val)
            score = 0.5*(e[10][0] + e[20][0])
            for h, (ep_, ev_) in e.items():
                rows.append(dict(regime=regime, model=f'koop-{variant}', lam=lam, horizon=h, pos_rmse=ep_, vel_rmse=ev_))
            if variant not in best or score < best[variant][0]: best[variant] = (score, m)
    for variant, (sc, m) in best.items(): models[f'koop-{variant}'] = m
    # Sélection RESTREINTE aux dictionnaires qui incluent l'historique d'entrée (flag "lag"=retards de u) :
    # un modèle Koopman sans aucune trace de la dynamique actionneur/retard fait de bonnes prédictions
    # passives (ce que mesure open_loop_errors) mais diverge en boucle fermée sous MPC agressif — voir
    # l'ablation 'koop-lin'/'koop-drag' dans les résultats. Ce n'est PAS un choix a posteriori sur le test :
    # c'est une contrainte de sélection décidée avant identification (un modèle sans mémoire d'entrée n'est
    # pas un candidat valable pour la commande), appliquée uniquement sur le split de validation.
    lag_variants = [v for v in VARIANT_FLAGS if VARIANT_FLAGS[v][1] == 1]
    sel = min(lag_variants, key=lambda v: best[v][0]); models['koop'] = best[sel][1]
    print(f"    dictionnaire sélectionné sur la VALIDATION parmi {lag_variants} : '{sel}' "
          f"(lam={models['koop']['lam']}, score 10/20 pas = {best[sel][0]:.3f} m)")
    print(f"    rho(A) koop = {max(abs(np.linalg.eigvals(models['koop']['A']))):.4f}")
    models['koopstab'] = stabilize_spectrum(models['koop'], RHO_MAX)
    ms = models['koopstab']
    print(f"    koopstab : rho {ms['rho_before']:.4f} -> {ms['rho_after']:.4f} "
          f"({ms['n_clipped']} valeur(s) propre(s) projetée(s) sur |lambda|={RHO_MAX})")
    for h, (ep_, ev_) in open_loop_errors(ms, val).items():
        rows.append(dict(regime=regime, model='koopstab', lam=np.nan, horizon=h, pos_rmse=ep_, vel_rmse=ev_))
    return models, sel, pd.DataFrame(rows)

In [ ]:
# =============================================================================
# 6. RÉGLAGE ÉQUITABLE (split 'tune') — PID par recherche aléatoire, MPC par grille (qscale, omega0)
# =============================================================================
def tune_score(df):
    return float(df.RMSE_pen.mean() + LAMBDA_U*df.ControlVariation.mean())

def tune_pid(regime, ctx):
    rng = np.random.default_rng(SEED+11); lg = lambda a, b: float(np.exp(rng.uniform(np.log(a), np.log(b))))
    cands = [dict(PID_DEFAULT)] + [dict(kp_xy=lg(6, 24), kd_xy=lg(3, 12), kp_z=lg(8, 32), kd_z=lg(4, 16),
                                        kR=lg(0.25, 2.0), kw=lg(0.04, 0.30), w_alpha=float(rng.uniform(0.5, 1.0)),
                                        omega=lg(2.5, 10)) for _ in range(N_TUNE_PID)]
    jobs = [dict(key=f'pid|{ci}|{i}', regime=regime, split='tune', trial=i, ctrl='pid_dob', override={'pid_params': c})
            for ci, c in enumerate(cands) for i in range(N_TUNE_TR)]
    df = run_jobs(jobs, desc='tune PID'); df['ci'] = df.key.str.split('|').str[1].astype(int)
    sc = df.groupby('ci').apply(tune_score); best = int(sc.idxmin())
    print(f"    PID : défaut score={sc[0]:.4f} ; meilleur (#{best}) score={sc[best]:.4f}")
    return cands[best], dict(default_score=float(sc[0]), best_score=float(sc[best]))

def tune_mpc(regime, ctx, model_key, ctrl=None, tag=None):
    ctrl = ctrl or f'{model_key}_mpc_dob'; tag = tag or model_key
    qss, rss, oms = ([0.3, 1.0], [1.0], [5.0]) if FAST_MODE else ([0.1, 0.3, 1.0], [1.0, 10.0], [3.0, 6.0])
    grid = list(itertools.product(qss, rss, oms))
    jobs = [dict(key=f'{tag}|{gi}|{i}', regime=regime, split='tune', trial=i, ctrl=ctrl,
                 override={'qscale': q, 'rscale': r, 'omega': o}) for gi, (q, r, o) in enumerate(grid) for i in range(N_TUNE_TR)]
    df = run_jobs(jobs, desc=f'tune {tag}'); df['gi'] = df.key.str.split('|').str[1].astype(int)
    sc = df.groupby('gi').apply(tune_score); best = int(sc.idxmin())
    fr = df[df.gi == best].fail.mean()
    print(f"    {tag} : meilleur (qscale={grid[best][0]}, rscale={grid[best][1]}, omega={grid[best][2]}) "
          f"score={sc[best]:.4f}, échecs tune={100*fr:.0f}%")
    return dict(qscale=grid[best][0], rscale=grid[best][1], omega=grid[best][2], score=float(sc[best]))

In [ ]:
# =============================================================================
# 7. PIPELINE : identification -> réglage -> GEL -> test / ablation
# =============================================================================
t_start = time.time()
CTX_ALL.clear(); val_tables, sel_dict = [], {}
for regime in ['mismatch', 'ideal']:
    models, sel, vt = identify_regime(regime)
    CTX_ALL[regime] = dict(regime=regime, models=models, pid_params=dict(PID_DEFAULT), mpc={}); sel_dict[regime] = sel
    val_tables.append(vt)
val_df = pd.concat(val_tables); val_df.to_csv(DATA_DIR/'validation_open_loop.csv', index=False)
print("\nErreur de prédiction en boucle ouverte (plantes de validation hors identification), position RMSE [m] :")
print(val_df[val_df.regime == 'mismatch'].assign(m=lambda d: d.model + '/' + d.lam.astype(str))
      .pivot_table(index='m', columns='horizon', values='pos_rmse').round(4).to_string())

# --- Réglage (régime 'mismatch' uniquement ; réutilisé tel quel pour 'ideal') ---
print("\n[TUNE] réglage sur le split 'tune' (jamais utilisé pour le test)")
ctx = CTX_ALL['mismatch']
best_pid, pid_scores = tune_pid('mismatch', ctx); ctx['pid_params'] = best_pid
tuned_mpc = {}
for mk in ['phys', 'physlag', 'koop', 'koopstab']:
    tuned_mpc[mk] = tune_mpc('mismatch', ctx, mk); ctx['mpc'][mk] = tuned_mpc[mk]
tuned_mpc['lqr_koopstab'] = tune_mpc('mismatch', ctx, 'koopstab', ctrl='lqr_koopstab_dob', tag='lqr_koopstab')
ctx['mpc']['lqr_koopstab'] = tuned_mpc['lqr_koopstab']
# Diagnostic isolé (option B) : LQR construit sur le modèle Koopman BRUT (non stabilisé). L'équation de
# Riccati discrète peut converger vers un gain stabilisant même si A a rho>1 en boucle ouverte, du moment
# que (A,B) est stabilisable — vérifié au préalable (rho(A)=1.015, rho(A-BK)=0.987 avec les poids par défaut).
# Sépare "stratégie de commande" (LQR vs MPC) de "qualité du modèle" (koop brut vs koopstab, corrompu par
# le clipping spectral de Schur) : lqr_koopstab_dob confondait les deux, lqr_koop_dob isole le premier facteur.
tuned_mpc['lqr_koop'] = tune_mpc('mismatch', ctx, 'koop', ctrl='lqr_koop_dob', tag='lqr_koop')
ctx['mpc']['lqr_koop'] = tuned_mpc['lqr_koop']
for mk in ['koop-lin', 'koop-drag', 'koop-lag', 'koop-full']:      # ablations : mêmes hyper-paramètres que 'koop'
    ctx['mpc'][mk] = dict(tuned_mpc['koop'])
CTX_ALL['ideal']['pid_params'] = dict(best_pid); CTX_ALL['ideal']['mpc'] = {k: dict(v) for k, v in ctx['mpc'].items()}

FROZEN = dict(seed=SEED, T_DUR=T_DUR, N_TEST=N_TEST, N_TEST_IDEAL=N_TEST_IDEAL, N_ABL=N_ABL, N_H=N_H,
              selected_dictionary=sel_dict, pid_params=best_pid, pid_tuning=pid_scores, mpc=tuned_mpc,
              fail_err=FAIL_ERR, rmse_cap=RMSE_CAP, lambda_u=LAMBDA_U, dob_d_max=DOB_D_MAX)
(PROJECT_ROOT/'frozen_hyperparameters.json').write_text(json.dumps(FROZEN, indent=2, default=float))
print("\nHyper-paramètres GELÉS et sauvegardés dans frozen_hyperparameters.json")
SIG = json.dumps(FROZEN, sort_keys=True, default=float)

# --- Test principal (split 'test') ---
print("\n[TEST] split 'test' : plantes et perturbations jamais vues")
jobs = [dict(key=f'{rg}|test|{i}|{c}', regime=rg, split='test', trial=i, ctrl=c)
        for rg, n in [('mismatch', N_TEST), ('ideal', N_TEST_IDEAL)] for i in range(n) for c in MAIN_CTRLS]
test_df = run_jobs(jobs, ckpt=DATA_DIR/'test_results.jsonl', sig=SIG, desc='test')
test_df.to_csv(DATA_DIR/'test_results.csv', index=False)

# --- Ablations (split 'abl', régime 'mismatch') ---
print("\n[ABL] ablations sur le split 'abl'")
ABL_CTRLS = MAIN_CTRLS + ['koop_mpc', 'physlag_mpc', 'koopstab_mpc', 'phys_mpc_dob',
                          'koop-lin_mpc_dob', 'koop-drag_mpc_dob', 'koop-lag_mpc_dob', 'koop-full_mpc_dob']
jobs = [dict(key=f'mismatch|abl|{i}|{c}|n1', regime='mismatch', split='abl', trial=i, ctrl=c)
        for i in range(N_ABL) for c in ABL_CTRLS]
jobs += [dict(key=f'mismatch|abl|{i}|{c}|n2', regime='mismatch', split='abl', trial=i, ctrl=c, noise=2.0, tag='noise2')
         for i in range(N_ABL) for c in MAIN_CTRLS]
abl_df = run_jobs(jobs, ckpt=DATA_DIR/'ablation_results.jsonl', sig=SIG, desc='ablation')
abl_df.to_csv(DATA_DIR/'ablation_results.csv', index=False)
print(f"\nCalcul total : {(time.time()-t_start)/60:.1f} min")

In [ ]:
# =============================================================================
# 8. STATISTIQUES, TABLES, FIGURES, ARCHIVE
# =============================================================================
rng_b = np.random.default_rng(SEED+99)

def summarize(df):
    rows = []
    for c, g in df.groupby('ctrl', sort=False):
        ok = g[g.fail == 0]
        rows.append(dict(controller=CTRL_LABEL.get(c, c), n=len(g), n_fail=int(g.fail.sum()),
                         success_pct=100*(1-g.fail.mean()),
                         RMSE_median=ok.RMSE.median(), RMSE_q25=ok.RMSE.quantile(.25), RMSE_q75=ok.RMSE.quantile(.75),
                         RMSE_pen_mean=g.RMSE_pen.mean(), P95_med=ok.P95Error.median(),
                         CV_med=g.ControlVariation.median(), CPU_ms=g.CPU_mean_ms.mean()))
    return pd.DataFrame(rows)

def paired(df, a, b):
    """Différence appariée par essai sur RMSE_pen (tous les essais, échecs imputés) : Wilcoxon + IC bootstrap."""
    da = df[df.ctrl == a].set_index('trial').RMSE_pen; db = df[df.ctrl == b].set_index('trial').RMSE_pen
    idx = da.index.intersection(db.index); d = (da.loc[idx]-db.loc[idx]).to_numpy()
    try: p = float(wilcoxon(d, alternative='two-sided').pvalue)
    except ValueError: p = np.nan
    bs = [np.median(rng_b.choice(d, len(d))) for _ in range(N_BOOT)]
    return dict(a=a, b=b, n=len(d), median_diff=float(np.median(d)), ci_lo=float(np.percentile(bs, 2.5)),
                ci_hi=float(np.percentile(bs, 97.5)), win_a_pct=float(100*np.mean(d < 0)), p=p)

def holm(ps):
    ps = np.array([1.0 if not np.isfinite(p) else p for p in ps]); o = np.argsort(ps); adj = np.empty_like(ps); run = 0.0
    for r, i in enumerate(o): run = max(run, (len(ps)-r)*ps[i]); adj[i] = min(1.0, run)
    return adj

def latex_table(path, df, caption, label, floats=3):
    f = lambda v: '--' if (isinstance(v, float) and not np.isfinite(v)) else (f'{v:.{floats}f}' if isinstance(v, float) else str(v))
    cols = [c.replace('_', r'\_') for c in df.columns]
    lines = [r'\begin{table}[H]', r'\centering', f'\\caption{{{caption}}}', f'\\label{{{label}}}',
             r'\begin{tabular}{'+'l'*len(cols)+'}', r'\toprule', ' & '.join(cols)+r' \\', r'\midrule']
    for _, r in df.iterrows(): lines.append(' & '.join(f(v).replace('_', r'\_') for v in r.values)+r' \\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    Path(path).write_text('\n'.join(lines), encoding='utf-8')

print("\n" + "="*78 + "\nRÉSULTATS TEST (les échecs sont comptés, jamais exclus)\n" + "="*78)
all_paired = []
for regime in ['mismatch', 'ideal']:
    d = test_df[test_df.regime == regime]
    if len(d) == 0: continue
    s = summarize(d); s.to_csv(DATA_DIR/f'summary_{regime}.csv', index=False)
    print(f"\n--- Régime '{regime}' ({d.trial.nunique()} essais/contrôleur) ---"); print(s.round(4).to_string(index=False))
    latex_table(TAB_DIR/f'table_main_{regime}.tex', s.round(4),
                f'Held-out test set, regime {regime}: failures counted (RMSE\\_pen = {RMSE_CAP} m for failed runs).', f'tab:main_{regime}')
    kind_tab = d.groupby(['kind', 'ctrl']).agg(fail=('fail', 'sum'), RMSE_pen=('RMSE_pen', 'mean')).reset_index()
    kind_tab.to_csv(DATA_DIR/f'by_kind_{regime}.csv', index=False)
    print("\n  par type de perturbation (échecs, RMSE pénalisé moyen) :")
    print(kind_tab.pivot(index='kind', columns='ctrl', values=['fail', 'RMSE_pen']).round(3).to_string())
    pairs = list(itertools.combinations(MAIN_CTRLS, 2))
    pr = [paired(d, a, b) for a, b in pairs]
    padj = holm([r['p'] for r in pr])
    for r, pa in zip(pr, padj): r['p_holm'] = float(pa); r['regime'] = regime
    all_paired += pr
    print(f"\n  comparaisons appariées (diff. médiane de RMSE_pen ; <0 => a meilleur ; Holm sur {len(pairs)} tests) :")
    print(pd.DataFrame(pr)[['a', 'b', 'n', 'median_diff', 'ci_lo', 'ci_hi', 'win_a_pct', 'p', 'p_holm']].round(4).to_string(index=False))
pd.DataFrame(all_paired).to_csv(DATA_DIR/'paired_tests.csv', index=False)

if len(abl_df):
    print("\n--- Ablations (split 'abl', régime mismatch) ---")
    a1 = summarize(abl_df[abl_df.tag == '']); a1.to_csv(DATA_DIR/'summary_ablation.csv', index=False); print(a1.round(4).to_string(index=False))
    latex_table(TAB_DIR/'table_ablation.tex', a1.round(4), 'Ablation: observer and dictionary.', 'tab:ablation')
    a2 = summarize(abl_df[abl_df.tag == 'noise2']); a2.to_csv(DATA_DIR/'summary_noise2.csv', index=False)
    print("\n--- Bruit de mesure x2 ---"); print(a2.round(4).to_string(index=False))

# --- Figures ---
mm = test_df[test_df.regime == 'mismatch']
if len(mm):
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    for k, (regime, a) in enumerate([('mismatch', ax[0]), ('ideal', ax[1])]):
        d = test_df[test_df.regime == regime]
        if len(d) == 0: continue
        data = [d[d.ctrl == c].RMSE_pen.values for c in MAIN_CTRLS]
        a.boxplot(data, labels=[CTRL_LABEL[c].replace(' ', '\n', 1) for c in MAIN_CTRLS], showmeans=True)
        a.tick_params(axis='x', labelsize=8)
        a.axhline(RMSE_CAP, ls=':', c='r'); a.set_title(f'Test — {regime}'); a.set_ylabel('RMSE pénalisé [m]'); a.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(FIG_DIR/'fig_test_boxplot.pdf'); plt.close(fig)
vp = val_df[(val_df.regime == 'mismatch')]
if len(vp):
    fig, ax = plt.subplots(figsize=(6, 4))
    for m_ in ['phys', 'physlag', 'koop-lin', 'koop-drag', 'koop-lag', 'koop-full']:
        g = vp[vp.model == m_].groupby('horizon').pos_rmse.min()
        ax.plot(g.index, g.values, 'o-', label=m_)
    ax.set_xlabel('horizon [pas]'); ax.set_ylabel('RMSE position, boucle ouverte [m]'); ax.set_yscale('log')
    ax.legend(); ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(FIG_DIR/'fig_open_loop.pdf'); plt.close(fig)
# trajectoire canonique : un essai 'combined' (vent+rafales+masse+CoM), 3 contrôleurs
try:
    spec = make_trial('test', 3, 'mismatch'); fig, ax = plt.subplots(figsize=(7, 4))
    for c in MAIN_CTRLS:
        L = run_trial(c, spec, CTX_ALL['mismatch'], keep_log=True)['log']
        ax.plot(L['t'], np.linalg.norm(L['p']-L['pref'], axis=1), label=CTRL_LABEL[c])
    ax.set_xlabel('t [s]'); ax.set_ylabel('erreur de position [m]'); ax.legend(); ax.grid(alpha=.25)
    ax.set_title(f"Essai test #3 ({spec['kind']}, {spec['family']})"); fig.tight_layout()
    fig.savefig(FIG_DIR/'fig_canonical_error.pdf'); plt.close(fig)
except Exception as e:
    print("figure canonique ignorée :", e)

val_df[val_df.regime == 'mismatch'].to_csv(DATA_DIR/'validation_open_loop_mismatch.csv', index=False)
(PROJECT_ROOT/'run_manifest.json').write_text(json.dumps(dict(
    frozen=FROZEN, files=sorted(str(p.relative_to(PROJECT_ROOT)) for p in PROJECT_ROOT.rglob('*') if p.is_file())),
    indent=2, default=float))
arch = shutil.make_archive(str(PROJECT_ROOT.parent/f'{PROJECT_ROOT.name}_results'), 'zip', root_dir=PROJECT_ROOT)
print(f"\nArchive : {Path(arch).resolve()}")

In [ ]:
# --- Aperçu rapide des figures + téléchargement de l'archive (Colab) ---
from IPython.display import Image, display
for f in sorted(FIG_DIR.glob('*.pdf')):
    print(f.name)
try:
    from google.colab import files
    files.download(arch)
except Exception as e:
    print("Téléchargement automatique indisponible ici ; récupérez le ZIP dans", arch)